In [ ]:
import os, sys

# Add repo root to sys.path so the in-repo mmengine/mmcv shims are found
_REPO_ROOT = os.path.dirname(os.path.abspath('.'))
if _REPO_ROOT not in sys.path:
    sys.path.insert(0, _REPO_ROOT)

print('Repo root:', _REPO_ROOT)
print('sys.path[0]:', sys.path[0])

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from mmengine.model import revert_sync_batchnorm
from mmseg.apis import init_model, inference_model
from mmseg.utils import get_classes, get_palette

import mmengine, mmcv, mmseg
print('mmengine:', mmengine.__version__)
print('mmcv    :', mmcv.__version__)
print('mmseg   :', mmseg.__version__)

In [ ]:
config_file     = '../configs/pspnet/pspnet_r50-d8_4xb2-40k_cityscapes-512x1024.py'
checkpoint_file = '../checkpoints/pspnet_r50-d8_512x1024_40k_cityscapes_20200605_003338-2966598c.pth'
img_path        = 'demo.png'

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

In [ ]:
# Download checkpoint if missing
import urllib.request
os.makedirs('../checkpoints', exist_ok=True)
if not os.path.isfile(checkpoint_file):
    url = ('https://download.openmmlab.com/mmsegmentation/v0.5/pspnet/'
           'pspnet_r50-d8_512x1024_40k_cityscapes/'
           'pspnet_r50-d8_512x1024_40k_cityscapes_20200605_003338-2966598c.pth')
    print(f'Downloading {url} ...')
    urllib.request.urlretrieve(url, checkpoint_file)
    print('Done.')
else:
    print('Checkpoint already present.')

In [ ]:
# Build model
model = init_model(config_file, checkpoint_file, device=device)
if not torch.cuda.is_available():
    model = revert_sync_batchnorm(model)
print('Model loaded:', type(model).__name__)

In [ ]:
# Run inference
result = inference_model(model, img_path)
seg_map = result.pred_sem_seg.data.squeeze(0).cpu().numpy().astype(np.int32)
print('Seg map shape:', seg_map.shape, ' labels:', sorted(np.unique(seg_map).tolist()))

In [ ]:
# Colour-code the segmentation map
dataset_meta = getattr(model, 'dataset_meta', {})
classes = dataset_meta.get('classes', get_classes('cityscapes'))
palette = dataset_meta.get('palette', get_palette('cityscapes'))

h, w = seg_map.shape
color_map = np.zeros((h, w, 3), dtype=np.uint8)
for label, color in enumerate(palette):
    color_map[seg_map == label] = color

# Blend with the original image
orig = np.array(Image.open(img_path).convert('RGB'))
if color_map.shape[:2] != orig.shape[:2]:
    color_map = np.array(Image.fromarray(color_map).resize(
        (orig.shape[1], orig.shape[0]), Image.NEAREST))
alpha = 0.5
blended = ((1 - alpha) * orig + alpha * color_map).clip(0, 255).astype(np.uint8)

In [ ]:
# Display results
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(orig);           axes[0].set_title('Original');        axes[0].axis('off')
axes[1].imshow(color_map);      axes[1].set_title('Segmentation');     axes[1].axis('off')
axes[2].imshow(blended);        axes[2].set_title('Overlay (α=0.5)'); axes[2].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class pixel coverage
total = seg_map.size
unique, counts = np.unique(seg_map, return_counts=True)
print(f"  {'Class':<30} {'Pixels':>8}  {'%':>6}")
print(f"  {'-'*30} {'-'*8}  {'-'*6}")
for lbl, cnt in sorted(zip(unique, counts), key=lambda x: -x[1]):
    name = classes[lbl] if lbl < len(classes) else f'class_{lbl}'
    print(f"  {name:<30} {cnt:>8}  {cnt/total*100:>5.1f}%")